# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [43]:
# %reload_ext dotenv
# %dotenv ../05_src/.secrets

from dotenv import load_dotenv
load_dotenv('../05_src/.secrets', override=True)

True

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [44]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../05_src/documents/Managing Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(len(docs))

13


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [45]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field, ConfigDict
import json

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

class ArticleAnalysis(BaseModel):
    model_config = ConfigDict(extra="forbid")  # ✅ forces additionalProperties: false
    Author: str
    Title: str
    Relevance: str = Field(description="<=A statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development")
    Summary: str = Field(description="Concise summary <= 1000 tokens")
    Tone: str
    InputTokens: int
    OutputTokens: int

In [46]:
TONE = "Bureaucratese"

DEVELOPER_INSTRUCTIONS = f"""
You are an assistant that extracts metadata and writes summaries.
Return output that strictly matches the provided JSON Schema.
The Summary MUST be written in the tone: {TONE}.
"""

USER_PROMPT = f"""
Analyze the following article text and produce the requested structured output.

<ArticleText>
{document_text}
</ArticleText>
"""

In [47]:
r = client.responses.create(
    model="gpt-4o-mini",
    input="Say OK"
)
print(r.output_text)

OK!


In [48]:
schema = ArticleAnalysis.model_json_schema()
print(schema)
response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": DEVELOPER_INSTRUCTIONS},
        {"role": "user", "content": USER_PROMPT},
    ],
text={
        "format": {
            "type": "json_schema",
            "name": "article_analysis",
            "schema": schema,
            "strict": True,
        }
    },
)

{'additionalProperties': False, 'properties': {'Author': {'title': 'Author', 'type': 'string'}, 'Title': {'title': 'Title', 'type': 'string'}, 'Relevance': {'description': '<=A statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development', 'title': 'Relevance', 'type': 'string'}, 'Summary': {'description': 'Concise summary <= 1000 tokens', 'title': 'Summary', 'type': 'string'}, 'Tone': {'title': 'Tone', 'type': 'string'}, 'InputTokens': {'title': 'Inputtokens', 'type': 'integer'}, 'OutputTokens': {'title': 'Outputtokens', 'type': 'integer'}}, 'required': ['Author', 'Title', 'Relevance', 'Summary', 'Tone', 'InputTokens', 'OutputTokens'], 'title': 'ArticleAnalysis', 'type': 'object'}


In [49]:
print(response.output_text)

{"Author":"Peter F. Drucker","Title":"Managing Oneself","Relevance":"This article is pertinent for AI professionals as it emphasizes self-management and understanding one’s strengths, which are crucial for navigating careers in a rapidly evolving technological landscape.","Summary":"The article delineates the necessity for individuals, particularly knowledge workers, to manage their own careers in an era where organizational loyalty is waning. Drucker posits that success hinges on self-awareness, particularly regarding one’s strengths, values, and preferred modes of operation. He advocates for 'feedback analysis' to discern personal strengths and urges professionals to seek environments where they can thrive. Drucker stresses the importance of aligning one's career trajectory with personal values to avoid frustration and promote effective performance. His insights also extend to the structuring of work relationships—emphasizing that understanding colleagues’ strengths and communication

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [50]:
print(type(response))
print(response)

<class 'openai.types.responses.response.Response'>
Response(id='resp_0aa1e3518e111e8100699a2900a80c81949a0f605c516c6012', created_at=1771710720.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_0aa1e3518e111e8100699a29024b6c819487d66d7f8dd8878c', content=[ResponseOutputText(annotations=[], text='{"Author":"Peter F. Drucker","Title":"Managing Oneself","Relevance":"This article is pertinent for AI professionals as it emphasizes self-management and understanding one’s strengths, which are crucial for navigating careers in a rapidly evolving technological landscape.","Summary":"The article delineates the necessity for individuals, particularly knowledge workers, to manage their own careers in an era where organizational loyalty is waning. Drucker posits that success hinges on self-awareness, particularly regarding one’s strengths, values, and preferred modes of operation. He advoc

In [51]:
import os
from openai import OpenAI
from deepeval.models import DeepEvalBaseLLM

AWS_BASE_URL = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"
AWS_GATEWAY_KEY = os.getenv("API_GATEWAY_KEY")

aws_client = OpenAI(
    base_url=AWS_BASE_URL,
    api_key="any-value",  # required but ignored by your gateway
    default_headers={"x-api-key": AWS_GATEWAY_KEY},
)

class AwsHostedJudge(DeepEvalBaseLLM):
    def __init__(self, model_name: str):
        self.model_name = model_name

    def load_model(self):
        return aws_client

    def generate(self, prompt: str) -> str:
        client = self.load_model()

        response = client.responses.create(
            model=self.model_name,
            input=[{"role": "user", "content": prompt}],
        )

        return response.output[0].content[0].text

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return f"AWS Hosted {self.model_name}"

In [52]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval

# Prepare Summary from response & 
json_str = response.output[0].content[0].text

# 2) Convert JSON string -> Python dict
analysis_json = json.loads(json_str)

# 3) Now you can access fields by key
summary = analysis_json["Summary"]

print("Keys:", list(analysis_json.keys()))
print("\n--- Summary preview ---\n")
print(summary[:800])

EVAL_MODEL = AwsHostedJudge("gpt-4o-mini")

#############################################
def evaluate_summary(original_text: str, summary: str) -> dict:

    # Create test case
    test_case = LLMTestCase(
        input=original_text,
        actual_output=summary,
    )

    # Summarization Metric
    summarization_metric = SummarizationMetric(
        model=EVAL_MODEL,
        assessment_questions=[
            "Does the summary capture the main thesis or purpose of the original text?",
            "Does the summary include the most important supporting points without major omissions?",
            "Does the summary avoid adding facts or details not present in the original text?",
            "Are key entities represented accurately?",
            "Does the summary avoid unsupported opinions?",
        ],
        threshold=0.5,
    )

    # Coherence Metric
    coherence_metric = GEval(
        name="Coherence",
        model=EVAL_MODEL,
        evaluation_steps=[
            "Check clarity.",
            "Check logical flow.",
            "Check ambiguity.",
            "Check focus.",
            "Check structure.",
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )

    # Tonality Metric
    tonality_metric = GEval(
        name="Tonality",
        model=EVAL_MODEL,
        evaluation_steps=[
            "Check neutral tone.",
            "Check tone consistency.",
            "Check professionalism.",
            "Check emotional exaggeration.",
            "Check bias.",
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )

    # Safety Metric
    safety_metric = GEval(
        name="Safety",
        model=EVAL_MODEL,
        evaluation_steps=[
            "Check PII exposure.",
            "Check harmful language.",
            "Check illegal instructions.",
            "Check risky claims.",
            "Check exploitative content.",
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )

    # Run all metrics
    metrics = [summarization_metric, coherence_metric, tonality_metric, safety_metric]

    for metric in metrics:
        metric.measure(test_case)

    # Collect results
    results = {
        "SummarizationScore": summarization_metric.score,
        "SummarizationReason": summarization_metric.reason,
        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason,
    }

    return results


Keys: ['Author', 'Title', 'Relevance', 'Summary', 'Tone', 'InputTokens', 'OutputTokens']

--- Summary preview ---

The article delineates the necessity for individuals, particularly knowledge workers, to manage their own careers in an era where organizational loyalty is waning. Drucker posits that success hinges on self-awareness, particularly regarding one’s strengths, values, and preferred modes of operation. He advocates for 'feedback analysis' to discern personal strengths and urges professionals to seek environments where they can thrive. Drucker stresses the importance of aligning one's career trajectory with personal values to avoid frustration and promote effective performance. His insights also extend to the structuring of work relationships—emphasizing that understanding colleagues’ strengths and communication preferences is vital for success. Furthermore, he discusses the rising trend toward


In [53]:
results = evaluate_summary(document_text, summary)

Output()

Output()

Output()

Output()

In [54]:
import textwrap
json_str = json.dumps(results, indent=2)
wrapped = textwrap.fill(json_str, width=100)

print(wrapped)

{   "SummarizationScore": 0.5555555555555556,   "SummarizationReason": "The score is 0.56 because
the summary contains contradictions regarding the emphasis on understanding colleagues\u2019
strengths, diverging from the original text's focus on self-knowledge. Additionally, extra
information is introduced about thriving environments and parallel careers that was not present in
the original text, leading to a less accurate portrayal of the main points.",   "CoherenceScore":
0.8,   "CoherenceReason": "The response demonstrates clarity and a logical flow, effectively
outlining Drucker's views on career management. The structure is coherent, with key ideas presented
in a sequential manner. However, there's minor ambiguity in the term 'feedback analysis' that could
be further clarified for a broader audience. Overall, it stays focused on the primary topic without
unnecessary distractions.",   "TonalityScore": 0.9,   "TonalityReason": "The article maintains a
neutral tone throughout, presen

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [55]:
# Step 1 — Create a simple enhancement prompt
enhancement_prompt = f"""
Use the context, the current summary, and the evaluation feedback to produce an improved summary.

Context:
{document_text}

Current Summary:
{summary}

Evaluation Feedback:
{results}

Instructions:
- Improve clarity and structure
- Fix issues mentioned in evaluation
- Do not add information not present in the context
- Keep it concise

Return ONLY the improved summary.
"""

In [56]:
#Step 2 — Generate the improved summary
enhanced_response = client.responses.create(
    model="gpt-4o-mini",
    input=[{"role": "user", "content": enhancement_prompt}],
    temperature=0.3
)

new_summary = enhanced_response.output[0].content[0].text

print(new_summary[:800])

The article emphasizes the importance of self-management for knowledge workers in an era of diminishing organizational loyalty. Drucker argues that success is rooted in self-awareness, particularly in understanding one's strengths, values, and preferred working styles. He recommends using 'feedback analysis' to identify personal strengths and suggests that individuals seek work environments that align with their values to enhance performance and satisfaction. Drucker also highlights the necessity of understanding colleagues’ strengths and communication styles to foster effective work relationships. Ultimately, he asserts that individuals must adopt a chief executive officer mindset to navigate their careers successfully, focusing on personal strengths while adapting to the evolving demands


In [57]:
# Step 3 — Evaluate the new summary using the SAME function
new_results = evaluate_summary(document_text, new_summary)

Output()

Output()

Output()

Output()

In [ ]:
json_str2 = json.dumps(new_results, indent=2)
new_wrapped = textwrap.fill(json_str2, width=100)

print(new_wrapped)

{   "SummarizationScore": 0.8571428571428571,   "SummarizationReason": "The score is 0.86 because
the summary effectively captures the main ideas from the original text, although it incorrectly
emphasizes understanding colleagues' strengths over individual strengths, which contradicts the
original message.",   "CoherenceScore": 0.9,   "CoherenceReason": "The response demonstrates high
clarity, presenting Drucker's arguments in a coherent manner. The logical flow is strong, moving
systematically from self-awareness to the practical application of feedback analysis, effectively
linking concepts. There is minimal ambiguity; the ideas are clearly defined and articulated. The
focus remains on self-management for knowledge workers, adhering closely to the topic. The structure
is well-organized, allowing readers to follow the argument easily. However, a slight improvement in
transitioning between points could enhance the overall cohesiveness.",   "TonalityScore": 0.9,
"TonalityReason": "The r

In [59]:
# Step 4 — Report results (basic comparison)
print("\n=== ORIGINAL SCORES ===")
print(wrapped)

print("\n=== NEW SCORES ===")
print(new_wrapped)


=== ORIGINAL SCORES ===
{   "SummarizationScore": 0.5555555555555556,   "SummarizationReason": "The score is 0.56 because
the summary contains contradictions regarding the emphasis on understanding colleagues\u2019
strengths, diverging from the original text's focus on self-knowledge. Additionally, extra
information is introduced about thriving environments and parallel careers that was not present in
the original text, leading to a less accurate portrayal of the main points.",   "CoherenceScore":
0.8,   "CoherenceReason": "The response demonstrates clarity and a logical flow, effectively
outlining Drucker's views on career management. The structure is coherent, with key ideas presented
in a sequential manner. However, there's minor ambiguity in the term 'feedback analysis' that could
be further clarified for a broader audience. Overall, it stays focused on the primary topic without
unnecessary distractions.",   "TonalityScore": 0.9,   "TonalityReason": "The article maintains a
neutra

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
